# StSN Builder — structure-based (foldseek)

Builds **Structure Similarity Networks (StSN)** from a directory of PDB / mmCIF files using foldseek.  
For sequence-based networks (mmseqs / FASTA), see `build_ssn.ipynb`.

The `pident` column in foldseek output represents TM-score × 100, so thresholds work the same way as in SSNs — threshold 0.5 means TM-score ≥ 0.5.

Run cells top to bottom, skipping optional ones as needed.

## Setup

In [15]:
import sys, logging
from pathlib import Path

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("stsn")

In [16]:
# Shared paths — edit these before running anything else
INPUT_PDB_DIR = Path("../data/structures/")    # directory of .pdb / .cif files
PREFIX        = "stsn_test"
OUTPUT_DIR    = Path("../results/stsn_run")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

---
## Step 1 — Redundancy reduction *(optional)*

Run **one** of the two cells below.

### 1a · Cluster with foldseek

In [8]:
from foldseek import easy_cluster

result = easy_cluster(
    pdb_dir        = INPUT_PDB_DIR,
    output_dir     = OUTPUT_DIR / "cluster",
    prefix         = PREFIX,
    min_tmscore    = 0.8,    # TM-score threshold for cluster membership (0–1)
    coverage       = 0.8,
    cov_mode       = 0,      # 0 = bidirectional, 1 = query, 2 = target
    alignment_type = 2,      # 0 = 3Di, 1 = TM-align, 2 = 3Di+AA
    log            = log,
)
rep_pdb_dir = INPUT_PDB_DIR   # foldseek clusters by ID; use same dir with rep list
cluster_tsv = result["cluster_tsv"]
print(f"cluster_tsv: {cluster_tsv}")

10:15:28  INFO      CMD  foldseek easy-cluster ../data/structures ../results/stsn_run/cluster/stsn_test ../results/stsn_run/cluster/tmp --min-seq-id 0.8 --cov-mode 0 -c 0.8 --alignment-type 2
10:20:45  INFO           easy-cluster ../data/structures ../results/stsn_run/cluster/stsn_test ../results/stsn_run/cluster/tmp --min-seq-id 0.8 --cov-mode 0 -c 0.8 --alignment-type 2 
10:20:45  INFO           
10:20:45  INFO           MMseqs Version:                     	10.941cd33
10:20:45  INFO           Substitution matrix                 	aa:3di.out,nucl:3di.out
10:20:45  INFO           Seed substitution matrix            	aa:3di.out,nucl:3di.out
10:20:45  INFO           Sensitivity                         	4
10:20:45  INFO           k-mer length                        	0
10:20:45  INFO           Target search mode                  	0
10:20:45  INFO           k-score                             	seq:2147483647,prof:2147483647
10:20:45  INFO           Max sequence length                 	65535


cluster_tsv: ../results/stsn_run/cluster/stsn_test_cluster.tsv


### 1b · Skip clustering — use full PDB directory

In [ ]:
rep_pdb_dir = INPUT_PDB_DIR
cluster_tsv = None
print(f"Using full PDB directory: {rep_pdb_dir}")

---
## Step 2 — All-vs-all structure similarity search (foldseek)

Run **one** of the two cells below.

### 2a · Run foldseek easy-search

In [9]:
from foldseek import easy_search

result = easy_search(
    pdb_dir        = rep_pdb_dir,
    output_dir     = OUTPUT_DIR / "search",
    prefix         = PREFIX,
    alignment_type = 0,    # 0 = 3Di, 1 = TM-align, 2 = 3Di+AA
    log            = log,
)
m8_path = result["m8"]
print(f"m8: {m8_path}")

10:30:04  INFO      CMD  foldseek easy-search ../data/structures ../data/structures ../results/stsn_run/search/stsn_test.m8 ../results/stsn_run/search/tmp --format-output query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits --alignment-type 0
11:08:00  INFO           ../results/stsn_run/search/stsn_test.m8 exists and will be overwritten
11:08:00  INFO           easy-search ../data/structures ../data/structures ../results/stsn_run/search/stsn_test.m8 ../results/stsn_run/search/tmp --format-output query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits --alignment-type 0 
11:08:00  INFO           
11:08:00  INFO           MMseqs Version:                    	10.941cd33
11:08:00  INFO           Seq. id. threshold                 	0
11:08:00  INFO           Coverage threshold                 	0
11:08:00  INFO           Coverage mode                      	0
11:08:00  INFO           Max reject                         	2147483647
11:08:00  INF

m8: ../results/stsn_run/search/stsn_test.m8


### 2b · Use existing .m8 file

In [19]:
m8_path = Path("/Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/stsn_run/search/stsn_test.m8")
print(f"Using existing m8: {m8_path}")

Using existing m8: /Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/stsn_run/search/stsn_test.m8


---
## Load similarity data

In [20]:
from plot_ssn import load_data

df = load_data(m8_path)
df.head()

11:42:01  INFO      START load similarity table /Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/stsn_run/search/stsn_test.m8
11:42:03  INFO      END   load similarity table /Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/stsn_run/search/stsn_test.m8: 2.74s
11:42:03  INFO      Loaded 7901459 rows from /Users/maartenboneschansker/Documents/bunch_of_code/SSN_v2/results/stsn_run/search/stsn_test.m8
11:42:03  INFO      START remove self-hits
11:42:04  INFO      END   remove self-hits: 0.60s
11:42:04  INFO      Removed self-hits where qseqid == sseqid: 7917 rows
11:42:04  INFO      After removing self-hits: 7893542 rows


Loaded data: 7893542 edges


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
1,A0A023ZY33,A0A1L6ZLP7,98.0,512,10,0,1,512,1,512,3.762000e-61,2829
2,A0A023ZY33,A0A5C0WMI7,98.0,512,10,0,1,512,1,512,1.690000e-61,2820
3,A0A023ZY33,G1E720,96.2,513,19,0,1,513,1,511,2.060000e-60,2807
4,A0A023ZY33,I2C651,88.2,495,58,0,20,513,26,520,1.449000e-59,2775
5,A0A023ZY33,A7Z5A2,87.1,512,66,0,3,513,1,512,2.646000e-60,2757


---
## Load annotations *(optional)*

Skip this cell to produce uncolored networks.  
Structure IDs in the m8 file are the PDB filenames without extension (e.g. `1abc` for `1abc.pdb`).

In [21]:
import pandas as pd
from plot_ssn import build_color_lookup, _read_meta

META_FILE    = Path("../data/GH43_full.tsv")
MAPPING_FILE = Path("/Users/maartenboneschansker/Documents/PhD/projects/cazyme_deep_dive/GH43_deep_dive/data/UniProt/GH43_UniProts.tsv")
ID_COL       = "GenBank"       # column in META_FILE; PDB nodes are UniProt IDs → mapped via id_map
COLOR_COLS   = ["Family", "Domain"]     # one plot is produced per column

meta = _read_meta(META_FILE, ID_COL)
print(f"Metadata: {len(meta)} rows  |  columns: {list(meta.columns)}")

# Load GenBank→UniProt mapping so structure nodes (UniProt IDs) get annotations
id_map = pd.read_csv(MAPPING_FILE, sep="\t", header=None, names=["genbank", "uniprot"])
print(f"ID map: {len(id_map)} GenBank→UniProt entries")

color_lookups = {}
for col in COLOR_COLS:
    color_lookups[col] = build_color_lookup(meta, ID_COL, col, clusters=None, id_map=id_map)
    print(f"  '{col}': {len(color_lookups[col])} annotated entries")

11:42:07  INFO      Extended lookup with 10189 UniProt keys


Metadata: 63591 rows  |  columns: ['Family', 'Domain', 'Species', 'GenBank', 'Source']
ID map: 10189 GenBank→UniProt entries
  'Family': 71815 annotated entries


11:42:08  INFO      Extended lookup with 10189 UniProt keys


  'Domain': 71815 annotated entries


In [22]:
# --- Debug: GenBank→UniProt mapping coverage ---
# Shows how many of the actual graph nodes resolve to an annotated UniProt entry.

# Collect all node IDs from the similarity data (these are the UniProt IDs in the graph)
all_node_ids = set(df["qseqid"].unique()) | set(df["sseqid"].unique())
print(f"Unique node IDs in similarity data : {len(all_node_ids)}")

# Inspect the id_map itself
print(f"\nid_map shape                        : {id_map.shape}")
print(f"id_map sample:\n{id_map.head(5).to_string()}")

# How many of those node IDs appear in the id_map's uniprot column?
uniprot_in_map = set(id_map["uniprot"].astype(str))
nodes_with_uniprot = all_node_ids & uniprot_in_map
print(f"\nNode IDs found in id_map (uniprot)  : {len(nodes_with_uniprot)} / {len(all_node_ids)}")
print(f"Node IDs NOT in id_map              : {len(all_node_ids) - len(nodes_with_uniprot)}")

# How many of those uniprot IDs also have a GenBank entry in the metadata?
for col in COLOR_COLS:
    lookup = color_lookups[col]
    covered     = sum(1 for n in all_node_ids if n in lookup)
    not_covered = len(all_node_ids) - covered
    print(f"\n[{col}] nodes with annotation         : {covered} / {len(all_node_ids)}")
    print(f"[{col}] nodes without annotation (Unknown): {not_covered}")

# Spot-check: show a few nodes that are NOT covered so you can inspect why
uncovered = [n for n in sorted(all_node_ids) if n not in color_lookups[COLOR_COLS[0]]]
print(f"\nFirst 10 uncovered node IDs: {uncovered[:10]}")

# Spot-check: show a few rows from id_map where the genbank is NOT in meta
genbank_in_meta = set(meta[ID_COL].astype(str))
missing_in_meta = id_map[~id_map["genbank"].astype(str).isin(genbank_in_meta)]
print(f"\nid_map rows where GenBank is absent from metadata: {len(missing_in_meta)} / {len(id_map)}")
print(missing_in_meta.head(5).to_string())

Unique node IDs in similarity data : 7917

id_map shape                        : (10189, 2)
id_map sample:
      genbank uniprot
0  AAA32682.1  P42256
1  AAA63610.1  P45982
2  AAB03089.1  Q45134
3  AAB08024.1  P49943
4  AAB08692.1  P77713

Node IDs found in id_map (uniprot)  : 7917 / 7917
Node IDs NOT in id_map              : 0

[Family] nodes with annotation         : 7917 / 7917
[Family] nodes without annotation (Unknown): 0

[Domain] nodes with annotation         : 7917 / 7917
[Domain] nodes without annotation (Unknown): 0

First 10 uncovered node IDs: []

id_map rows where GenBank is absent from metadata: 0 / 10189
Empty DataFrame
Columns: [genbank, uniprot]
Index: []


---
## Step 3a — Structure Similarity Network (StSN)

Full graph — all edges above the TM-score threshold.

In [23]:
import json
from plot_ssn import create_graph, exclude_singleton_components, get_layout, plot_ssn, graph_stats

THRESHOLDS         = [0.4, 0.6]       # TM-score thresholds (0–1)
NODE_SIZE          = 5
LAYOUT             = "component_grid" # "component_grid" | "fr" | "lgl" | "kk"
EXCLUDE_SINGLETONS = True

_lookups    = locals().get("color_lookups", {})
_meta       = locals().get("meta", None)
_id_col     = locals().get("ID_COL", None)
_color_cols = list(_lookups.keys()) or [None]

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
stsn_stats = []

for thresh in THRESHOLDS:
    thresh_pident = thresh * 100 if thresh < 1 else thresh
    df_thresh = df[(df["pident"] >= thresh_pident) & (df["qseqid"] != df["sseqid"])].copy()

    g = create_graph(df_thresh)
    if g is None or g.vcount() == 0:
        print(f"t={thresh}: no nodes, skipping."); continue

    if EXCLUDE_SINGLETONS:
        g, _ = exclude_singleton_components(g)
        if g is None or g.vcount() == 0:
            print(f"t={thresh}: empty after singleton removal, skipping."); continue

    print(f"t={thresh}: {g.vcount()} nodes, {g.ecount()} edges")
    layout_coords = get_layout(g, LAYOUT)
    stsn_stats.append(graph_stats(g, thresh, metadata=_meta, id_col=_id_col))

    for col in _color_cols:
        lookup   = _lookups.get(col) or {n: "Unknown" for n in g.vs["name"]}
        col_lbl  = col if col else "uncolored"
        out_file = OUTPUT_DIR / "plots" / f"{PREFIX}_stsn_t{thresh}_{col_lbl}.html"
        plot_ssn(g, str(out_file), col, thresh, str(m8_path), lookup, NODE_SIZE,
                 layout_coords=layout_coords, metadata=_meta, id_col=_id_col)

stats_path = OUTPUT_DIR / "plots" / f"{PREFIX}_stsn_stats.json"
stats_path.write_text(json.dumps(stsn_stats, indent=2))
print(f"\nStats → {stats_path}")
pd.DataFrame(stsn_stats)

11:42:19  INFO      START deduplicate undirected edges
11:42:19  INFO      Deduplicated reciprocal/parallel edges: 1041229 -> 529066
11:42:19  INFO      END   deduplicate undirected edges: 0.40s
11:42:19  INFO      START create edge list
11:42:20  INFO      END   create edge list: 0.06s
11:42:20  INFO      START build igraph
11:42:20  INFO      END   build igraph: 0.12s
11:42:20  INFO      START attach edge pident weights
11:42:20  INFO      END   attach edge pident weights: 0.01s
11:42:20  INFO      START compute component_grid layout (7816 nodes, 529066 edges)
11:42:28  INFO      END   compute component_grid layout (7816 nodes, 529066 edges): 8.30s


t=0.4: 7816 nodes, 529066 edges


11:42:28  INFO      START prepare node colors for Family
11:42:28  INFO      END   prepare node colors for Family: 0.00s
11:42:28  INFO      START build edge trace for Family
11:42:31  INFO      END   build edge trace for Family: 3.28s
11:42:31  INFO      START build node traces for Family (discrete)
11:42:46  INFO      END   build node traces for Family (discrete): 14.91s
11:42:46  INFO      START assemble figure for Family
11:42:50  INFO      END   assemble figure for Family: 3.33s
11:42:50  INFO      START write HTML ../results/stsn_run/plots/stsn_test_stsn_t0.4_Family.html
11:42:51  INFO      END   write HTML ../results/stsn_run/plots/stsn_test_stsn_t0.4_Family.html: 1.39s
11:42:51  INFO      Saved: ../results/stsn_run/plots/stsn_test_stsn_t0.4_Family.html
11:42:51  INFO      START write PNG ../results/stsn_run/plots/stsn_test_stsn_t0.4_Family.png


  Saved: ../results/stsn_run/plots/stsn_test_stsn_t0.4_Family.html


11:43:14  INFO      END   write PNG ../results/stsn_run/plots/stsn_test_stsn_t0.4_Family.png: 22.55s
11:43:14  INFO      Saved: ../results/stsn_run/plots/stsn_test_stsn_t0.4_Family.png
11:43:14  INFO      START prepare node colors for Domain
11:43:14  INFO      END   prepare node colors for Domain: 0.01s
11:43:14  INFO      START build edge trace for Domain


...

11:44:20  INFO      END   build edge trace for Domain: 0.57s
11:44:20  INFO      START build node traces for Domain (discrete)
11:44:33  INFO      END   build node traces for Domain (discrete): 12.97s
11:44:33  INFO      START assemble figure for Domain
11:44:34  INFO      END   assemble figure for Domain: 0.62s
11:44:34  INFO      START write HTML ../results/stsn_run/plots/stsn_test_stsn_t0.6_Domain.html
11:44:34  INFO      END   write HTML ../results/stsn_run/plots/stsn_test_stsn_t0.6_Domain.html: 0.25s
11:44:34  INFO      Saved: ../results/stsn_run/plots/stsn_test_stsn_t0.6_Domain.html
11:44:34  INFO      START write PNG ../results/stsn_run/plots/stsn_test_stsn_t0.6_Domain.png


  Saved: ../results/stsn_run/plots/stsn_test_stsn_t0.6_Domain.html


11:44:38  INFO      END   write PNG ../results/stsn_run/plots/stsn_test_stsn_t0.6_Domain.png: 4.24s
11:44:38  INFO      Saved: ../results/stsn_run/plots/stsn_test_stsn_t0.6_Domain.png


  Saved: ../results/stsn_run/plots/stsn_test_stsn_t0.6_Domain.png

Stats → ../results/stsn_run/plots/stsn_test_stsn_stats.json


,threshold,nodes,edges,n_components,largest_component,singletons
0,0.4,7816,529066,90,7001,0
1,0.6,6815,93425,690,588,0


---
## Step 3b — Minimum Spanning Tree (StSN-MST) *(optional)*

Reduces each connected component to its spanning tree — keeps only the highest-TM-score edge per pair.  
Often the preferred view for structure networks as it avoids edge clutter.

In [13]:
from plot_mst import (
    build_mst_graph,
    exclude_singleton_components as mst_exclude_singletons,
    get_layout as mst_get_layout,
    plot_mst,
    graph_stats as mst_graph_stats,
)

THRESHOLDS         = [0.5, 0.7]       # TM-score thresholds (0–1)
NODE_SIZE          = 5
LAYOUT             = "component_grid"
EXCLUDE_SINGLETONS = True

_lookups    = locals().get("color_lookups", {})
_meta       = locals().get("meta", None)
_id_col     = locals().get("ID_COL", None)
_color_cols = list(_lookups.keys()) or [None]

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
mst_stats = []

for thresh in THRESHOLDS:
    thresh_pident = thresh * 100 if thresh < 1 else thresh
    df_thresh = df[(df["pident"] >= thresh_pident) & (df["qseqid"] != df["sseqid"])].copy()

    g = build_mst_graph(df_thresh)
    if g is None or g.vcount() == 0:
        print(f"t={thresh}: no nodes, skipping."); continue

    if EXCLUDE_SINGLETONS:
        g, _ = mst_exclude_singletons(g)
        if g is None or g.vcount() == 0:
            print(f"t={thresh}: empty after singleton removal, skipping."); continue

    print(f"t={thresh}: {g.vcount()} nodes, {g.ecount()} MST edges")
    layout_coords = mst_get_layout(g, LAYOUT)
    mst_stats.append(mst_graph_stats(g, thresh, metadata=_meta, id_col=_id_col))

    for col in _color_cols:
        lookup   = _lookups.get(col) or {n: "Unknown" for n in g.vs["name"]}
        col_lbl  = col if col else "uncolored"
        out_file = OUTPUT_DIR / "plots" / f"{PREFIX}_mst_t{thresh}_{col_lbl}.html"
        plot_mst(g, str(out_file), col, thresh, str(m8_path), lookup, NODE_SIZE,
                 layout_coords=layout_coords, metadata=_meta, id_col=_id_col)

stats_path = OUTPUT_DIR / "plots" / f"{PREFIX}_mst_stats.json"
stats_path.write_text(json.dumps(mst_stats, indent=2))
print(f"\nStats → {stats_path}")
pd.DataFrame(mst_stats)

11:25:07  INFO      START deduplicate undirected edges
11:25:08  INFO      Deduplicated edges: 535253 -> 269574
11:25:08  INFO      END   deduplicate undirected edges: 0.23s
11:25:08  INFO      START build node index
11:25:08  INFO      END   build node index: 0.02s
11:25:08  INFO      START build sparse distance matrix
11:25:08  INFO      END   build sparse distance matrix: 0.02s
11:25:08  INFO      START compute minimum spanning tree (forest)
11:25:08  INFO      END   compute minimum spanning tree (forest): 0.02s
11:25:08  INFO      START extract MST edges
11:25:08  INFO      MST has 7210 edges (from 269574 input edges)
11:25:08  INFO      END   extract MST edges: 0.00s
11:25:08  INFO      START build igraph from MST edges
11:25:08  INFO      END   build igraph from MST edges: 0.00s
11:25:08  INFO      START compute component_grid layout (7526 nodes, 7210 edges)


t=0.5: 7526 nodes, 7210 MST edges


11:25:09  INFO      END   compute component_grid layout (7526 nodes, 7210 edges): 1.71s
11:25:09  INFO      START prepare node colors for Family
11:25:09  INFO      END   prepare node colors for Family: 0.00s
11:25:09  INFO      START build edge trace for Family
11:25:09  INFO      END   build edge trace for Family: 0.05s
11:25:09  INFO      START build node traces for Family (discrete)
11:25:25  INFO      END   build node traces for Family (discrete): 15.21s
11:25:25  INFO      START assemble figure for Family
11:25:25  INFO      END   assemble figure for Family: 0.06s
11:25:25  INFO      START write HTML ../results/stsn_run/plots/stsn_test_mst_t0.5_Family.html
11:25:25  INFO      END   write HTML ../results/stsn_run/plots/stsn_test_mst_t0.5_Family.html: 0.04s
11:25:25  INFO      Saved: ../results/stsn_run/plots/stsn_test_mst_t0.5_Family.html
11:25:25  INFO      START write PNG ../results/stsn_run/plots/stsn_test_mst_t0.5_Family.png


  Saved: ../results/stsn_run/plots/stsn_test_mst_t0.5_Family.html


11:25:25  INFO      END   write PNG ../results/stsn_run/plots/stsn_test_mst_t0.5_Family.png: 0.81s
11:25:25  INFO      Saved: ../results/stsn_run/plots/stsn_test_mst_t0.5_Family.png
11:25:25  INFO      START prepare node colors for Domain
11:25:25  INFO      END   prepare node colors for Domain: 0.00s
11:25:25  INFO      START build edge trace for Domain
11:25:26  INFO      END   build edge trace for Domain: 0.05s
11:25:26  INFO      START build node traces for Domain (discrete)


...

11:26:03  INFO      END   build node traces for Domain (discrete): 10.44s
11:26:03  INFO      START assemble figure for Domain
11:26:03  INFO      END   assemble figure for Domain: 0.04s
11:26:03  INFO      START write HTML ../results/stsn_run/plots/stsn_test_mst_t0.7_Domain.html
11:26:03  INFO      END   write HTML ../results/stsn_run/plots/stsn_test_mst_t0.7_Domain.html: 0.03s
11:26:03  INFO      Saved: ../results/stsn_run/plots/stsn_test_mst_t0.7_Domain.html
11:26:03  INFO      START write PNG ../results/stsn_run/plots/stsn_test_mst_t0.7_Domain.png


  Saved: ../results/stsn_run/plots/stsn_test_mst_t0.7_Domain.html


11:26:03  INFO      END   write PNG ../results/stsn_run/plots/stsn_test_mst_t0.7_Domain.png: 0.57s
11:26:03  INFO      Saved: ../results/stsn_run/plots/stsn_test_mst_t0.7_Domain.png


  Saved: ../results/stsn_run/plots/stsn_test_mst_t0.7_Domain.png

Stats → ../results/stsn_run/plots/stsn_test_mst_stats.json


,threshold,nodes,mst_edges,n_components,largest_component,singletons
0,0.5,7526,7210,316,2817,0
1,0.7,5571,4659,912,282,0


---
## Results

In [ ]:
from IPython.display import display, HTML

html_files = sorted((OUTPUT_DIR / "plots").glob("*.html"))
items = "".join(f'<li><a href="{f.resolve()}" target="_blank">{f.name}</a></li>' for f in html_files)
display(HTML(f"<ul>{items}</ul>") if items else HTML("<i>No plots found — run cells above first.</i>"))